# **肌電訊號量測實作單元(三)：教練這一組我真的不行了！不信你看我的肌電訊號！**

> 在這次的實驗中，鐵克將會帶大家用FFT觀察肌電訊號的頻域資訊。

> 並嘗試用MF和MPF來分析自己的肌電訊號，透過客觀資訊來評估你的肌肉是否確實感到疲乏囉！

# 0. 先收錄一些會用到的公式和函式吧！

In [ ]:
#@title
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
def butter_bandpass(low_cut, high_cut, fs, order=3):
    """
    Design band pass filter.

    Args:
        - low_cut  (float) : the low cutoff frequency of the filter.
        - high_cut (float) : the high cutoff frequency of the filter.
        - fs       (float) : the sampling rate.
        - order      (int) : order of the filter, by default defined to 5.
    """
    # calculate the Nyquist frequency
    nyq = 0.5 * fs

    # design filter
    low = low_cut / nyq
    high = high_cut / nyq
    b, a = butter(order, [low, high], btype='band')

    # returns the filter coefficients: numerator and denominator
    return b, a 

from scipy.signal import butter, lfilter
def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y

# 1.先將自己持續出力的肌電訊號丟進程式庫吧！

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
df = pd.read_csv('範例肌電訊號2.txt')  #記得修改('xxxx')裡面的檔名唷！
df.info()
data = np.array(df)
data2 = data[0 : , 0]

# 2. 畫出自己尚未做任何處理的肌電訊號長什麼樣子吧！

In [ ]:
#稍微設定一下畫布資訊！
start_time = 2 #單位：秒
end_time = 8 #單位：秒

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data2[start_time*1000:end_time*1000], #取用其中一段肌電訊號
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="其中一段肌電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

# 3. 根據論文描述，用帶通濾波器將肌電訊號的頻帶範圍(20Hz~400Hz)給擷取出來吧！

In [ ]:
data3 = butter_bandpass_filter(data2, 20, 400, 1000)

start_time = 2 #單位：秒
end_time = 8 #單位：秒

#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data3[start_time*1000:end_time*1000], #取用其中一段肌電訊號
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="濾波後的肌電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

# 4. 將濾波完的肌電訊號透過FFT取得MF和MPF吧！

> 為了觀察出力期間的MF和MPF的變化，我們試著以2秒(2000毫秒)的資料為單位，做FFT取得期間的頻域資訊。

> 取得頻域資訊後，試著做出該段資料得MF(median frequency)和MPF(middle power frequency)吧！

In [ ]:
N = 2000 #往前抓2000個資料做FFT (以1000Hz取樣頻率的資料，1秒的資料為單位即為1000個資料)
T = 0.001 #每個資料得間隔時間為0.001秒 (以1000Hz取樣頻率的資料，資料間隔即為0.001秒)
x = np.linspace(0.0, N*T, N, endpoint=False)
yf = fft(data3[1:N])
xf = fftfreq(N, T)[1:N//2] 
yf_abs = np.abs(yf[1:N//2]) 
fig = go.Figure()
fig.add_trace( go.Scatter(x=xf, y=2.0/N * yf_abs))
fig.update_layout(
    title="肌電訊號取FFT後的頻域能量資料",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
#根據公式，做MF最重要的第一件事就是要知道所有頻率能量加總是多少！
#之後只要再重新累加，就很容易能取得中位數頻率能量的位置囉！

halfSum = 0 #預備做累加至一半能量總和
MF = 0 #計算累加至哪個頻段抵達MF位置
fullSum = np.sum(yf_abs) #所有能量加總

for i in yf_abs:
  if(halfSum < fullSum/2): #累加至halfSum會抵達MF位置(一半能量總和)
    halfSum = halfSum + i
    MF += 1
  else:
    break
whole_MF = MF
print('整段資料的median frequency為', whole_MF,'赫茲',)    

In [ ]:
#根據公式，做MPF即是把所有頻率的能量對頻率本身做加權平均！

fullSum = np.sum(yf_abs) #所有能量加總
frequency_pointer = 0 #用來取得頻率資訊的移動指標
MPF = 0 #預備累加至能量總和

for i in yf_abs:
  MPF = MPF + i*xf[frequency_pointer]
  frequency_pointer += 1

whole_MPF = MPF / fullSum

print('整段資料的middle power frequency為', whole_MPF,'赫茲',)    


# 5. 了解FFT、MF和MPF的做法後，試著分段處理肌電訊號吧！

> 以2秒的肌電訊號實作一次FFT，並計算該段時間的MF和MPF！

> 實作一次後再往後平移0.5秒就實作一次，重複計算來觀察整段肌電訊號的MF和MPF的變化吧！

In [ ]:
loop_time = 500
loop_number = int((data3.size-N)/loop_time) #計算每0.5秒實作一次，共可做幾次呢
loop_counter = 0 #每完成一次0.5秒的實作後+1

N = 2000 #往前抓2000個資料做FFT (以1000Hz取樣頻率的資料，1秒的資料為單位即為1000個資料)
T = 0.001 #每個資料得間隔時間為0.001秒 (以1000Hz取樣頻率的資料，資料間隔即為0.001秒)
x = np.linspace(0.0, N*T, N, endpoint=False)
xf = fftfreq(N, T)[1:N//2] 

MPF_set = np.zeros(loop_number) #存放每次0.5秒實作後的MPF數據
MF_set = np.zeros(loop_number)  #存放每次0.5秒實作後的MF數據


while loop_counter < loop_number:
  
  #-------------實作FFT---------------#
  yf = fft(data3[1+(loop_time)*(loop_counter):N+(loop_time)*(loop_counter)])
  yf_abs = np.abs(yf[1:N//2]) 

  #-------------實作MF---------------#
  halfSum = 0 #預備做累加至一半能量總和
  MF = 0 #計算累加至哪個頻段抵達MF位置
  fullSum = np.sum(yf_abs) #所有能量加總

  for i in yf_abs:
    if(halfSum < fullSum/2): #累加至halfSum會抵達MF位置(一半能量總和)
      halfSum = halfSum + i
      MF += 1
    else:
      break
  MF_set[loop_counter] = MF


  #-------------實作MPF---------------#
  frequency_pointer = 0 #用來取得頻率資訊的移動指標
  MPF = 0 #預備累加至能量總和

  for i in yf_abs:
    MPF = MPF + i*xf[frequency_pointer]
    frequency_pointer += 1

  MPF = MPF / fullSum

  MPF_set[loop_counter] = MPF

  #-------------完成一組0.5秒實作---------------#
  #---------------開始換下一組！---------------#
  loop_counter+=1
  #....
  #............
  #....................
  #........................
  #------直到loop_counter等於loop_number-------#

print("成功完成整段肌電訊號的MF與MPF分析！")

# 6. 來看看努力分析過後的MF和MPF的變化如何，並依此來評估肌肉疲勞程度吧！

In [ ]:
#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=MF_set, #描繪出MF的變化
    mode='lines',
    name='MF Plot',
)
)
fig.add_trace(go.Scatter(
    y=MPF_set, #描繪出MPF的變化
    mode='lines',
    name='MPF Plot',
)
)
fig.update_layout(
    title="整段出力過程肌電訊號的MF與MPF的變化",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)

part_MF_max = 0
part_MF_min = 1000
part_MPF_max = 0
part_MPF_min = 1000

for i in MF_set:
  if i>part_MF_max:
    part_MF_max = i
  if i<part_MF_min:
    part_MF_min = i
for i in MPF_set:
  if i>part_MPF_max:
    part_MPF_max = i
  if i<part_MPF_min:
    part_MPF_min = i

print("整段肌電的MF：",int(whole_MF))
print("整段肌電的MPF：",int(whole_MPF))
print("片段肌電的最大MF：",int(part_MF_max))
print("片段肌電的最小MF：",int(part_MF_min))
print("片段肌電的最大MPF：",int(part_MPF_max))
print("片段肌電的最小MPF：",int(part_MPF_min))
fig.show()

# 章節總結：
---

1.   肌電訊號的頻域分析雖在學理上有明確的指引，但實際要讓肌肉感到疲乏，並在肌電訊號上反應，並非透過單次的出力就能達到明顯的成效。
2.   MF和MPF雖有著完全不同的公式，但其物理意義皆是反應中低頻的能量變化，因此兩種指標的升降趨勢雷同。
3.   有沒有可能在多次使用肌肉後，將前幾次的量測結果與後幾次的相比，來取得更明顯的肌肉疲乏證據呢(如較低的MF平均值或MPF平均值)？

# 恭喜完成本章節課程



> 有需要的同學可以使用密碼搭配書籍網站取得課程證書

> 證書密碼：「yutechEMG2」！